# FR 2016 Singles P2a — ACTIVE-DEVELOPMENT notebook (v2)

> ## ⚠️ Active development — provisional, not a certification artifact
>
> - This is the **active-development** copy of the frozen region-live P2a pipeline. `fr_singles_pipeline_v1.ipynb` is the frozen v1 checkpoint; edit and run **here**, not there.
> - **P2a region-live is PROVISIONAL** (target negLL 19053.4655): not accepted, not safe for inference, manuscript results, or certified welfare.
> - The **certified 47-parameter pooled model** (`joint_pooled_v1_bll0_tlmpin`, negLL 238504.6360973987; France 2015–2017; synthetic-recovery certified; real-data Hessian PD; clustered on `idorighh`) **remains the formal JMP baseline** and is untouched by this notebook.
> - **Notebook results are NOT certification artifacts.** Every artifact this notebook writes lands under `NOTEBOOK_OUTPUT_ROOT` (`outputs/p2a_singles2016/notebook_dev_v2/`) and must never be treated as production.
> - **Sole authorized production-output exception (operator-gated):** the immutable **geometry freeze** of the in-memory `draws_p2a` frame to `outputs/p2a_singles2016/region_live_v1/inputs/` — the production runner's G-0 frozen input (manager decision D-1 v2). It runs **only** when `EXPORT_PRODUCTION_GEOMETRY=True` is set in the controls cell, never rewrites an existing valid freeze, and authorizes **nothing else**: estimation, inference, post-estimation, and welfare writes to `region_live_v1/` remain prohibited from this notebook.
> - **Notebook welfare outputs are NOT safe for manuscript use.**
> - **The strict production verdict comes from `MNL/scripts/p2a/`** (the JMP-specific rebuild + strict-diagnostics code), not from this notebook. This notebook produces preliminary evidence only.
>
> Governance: `docs/France_case/P2a/FR_P2a_region_live_manager_decisions_v2.md` (canonical) and the region-live production rebuild plan. See the integration addendum `FR_P2a_region_live_notebook_integration_addendum_v1.md`.


# FR 2016 Singles RURO — Production Pipeline (P2a certified B-pool draws, region-live)

Linear, re-runnable singles pipeline distilled from the exploratory `fr_data_walkthrough.ipynb`
(kept as the historical record — **do not** delete it). Only the *final* path survives here:
certified B-pool draws (D1 hours mixture, W1 occupation-conditional wages, empirical occupation,
pi0=0.10, seed 2026), bsa00yn_a=1 deterministic pricing, bpool band convention, region/urbanisation/
gsur revived onto the engine-ready frame (§12b), and the fit with the 10 couples/year params pinned
and the **occupation block free**.

**Regression anchor:** the pipeline reproduces the region-live P2a negLL **19053.4655**. (The earlier
region-*dead* fit, 19071.6562, left `beta_E` absorbing regional variation; reviving the regional
covariates lowers negLL by ~18 and moves `beta_E` −4.31 → −2.90.)

**Welfare (§19):** the final section produces V_i^IS and the W1/W3/W4/W6 welfare-measure family plus
the survey-weighted inequality battery on this baseline (P3-1), by running the two `scripts/welfare/`
tools and verifying the outputs are region-live.

**SKIP_PRICING** — EUROMOD pricing is guarded. With `SKIP_PRICING=True` the pricing cells load the
cached parquets in `fr_singles_pricing_p2a/` instead of calling EUROMOD; set `False` to reprice.
A dropped-cells appendix at the end lists every exploratory/superseded cell removed and why.

In [ ]:
# === v2 execution controls + isolated output root ==============================
# Active-development copy. Every expensive stage is OFF by default — flip a flag to run
# that stage. Every notebook-generated artifact is redirected beneath NOTEBOOK_OUTPUT_ROOT
# so v2 can never overwrite the frozen v1 / production P2a artifacts.
from pathlib import Path

RUN_PRICING         = False   # EUROMOD pricing of the draws (else: load the shared read cache)
RUN_ESTIMATION      = False   # L-BFGS-B fit + freeze/adapter
RUN_INFERENCE       = False   # cluster-robust sandwich SEs (35 free non-bound params)
RUN_POST_ESTIMATION = False   # styled post-estimation report
RUN_WELFARE         = False   # V_i^IS + W-measure family (PROVISIONAL — see banner)

# OPERATOR GATE (production exception): freeze the in-memory draws_p2a geometry into
# the production inputs folder consumed by the scripts/p2a runner (G-0 frozen input).
# Sole authorized production write -- see the banner and addendum s15. Default False.
EXPORT_PRODUCTION_GEOMETRY = False

NOTEBOOK_OUTPUT_ROOT = Path('outputs/p2a_singles2016/notebook_dev_v2')   # created after chdir (setup cell)

print('v2 controls:', dict(RUN_PRICING=RUN_PRICING, RUN_ESTIMATION=RUN_ESTIMATION,
      RUN_INFERENCE=RUN_INFERENCE, RUN_POST_ESTIMATION=RUN_POST_ESTIMATION, RUN_WELFARE=RUN_WELFARE,
      EXPORT_PRODUCTION_GEOMETRY=EXPORT_PRODUCTION_GEOMETRY))
print('NOTEBOOK_OUTPUT_ROOT =', NOTEBOOK_OUTPUT_ROOT)


In [ ]:
# ── Setup, flags, and raw read ──────────────────────────────────────────────
import sys, time, json
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', 80); pd.set_option('display.width', 200)

# Pricing cells load the shared cache unless RUN_PRICING is set (see the controls cell).
SKIP_PRICING = not RUN_PRICING

# repo root + the certified draw modules (they cross-import by bare name).
MNL_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'scripts/bpool/build_bpool_singles.py').exists()), None)
assert MNL_ROOT, 'MNL repo root not found'
import os; os.chdir(MNL_ROOT)   # all relative artifacts (pricing cache, outputs/, parquets) resolve at repo root
NOTEBOOK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)   # v2 isolated artifact root
for sub in ('scripts/bpool', 'scripts/pilot'):
    d = str(MNL_ROOT / sub)
    if d not in sys.path: sys.path.insert(0, d)

FR_RAW = Path('C:/Users/hisham/MNL/EUROMOD-STORAGE/Data/FR/FR_2016_a3.txt')
raw = pd.read_csv(FR_RAW, sep='\t')
print('raw shape   :', raw.shape, '| households:', raw['idhh'].nunique(), '| persons:', len(raw))

## 1. Schema check — confirm the EUROMOD-standard + FR-specific columns are present

In [ ]:
# key model inputs + the FR-specific columns the pipeline depends on
_need = ['idhh','idperson','idpartner','dag','dgn','les','lhw','deh','loc','yivwg','yem','yse']
_fr   = ['drgn1','drgur','drgmd','drgru','yem00','yemxp','lma','liwwh','dew','dey','bsa00','bsa00_s']
print('required present :', {c: (c in raw.columns) for c in _need})
print('FR-specific      :', {c: (c in raw.columns) for c in _fr})
assert all(c in raw.columns for c in _need), 'missing a required EUROMOD input column'

## 2. Eligibility config (FR-mirror constants, from the DE adapter)

In [ ]:
CONFIG = dict(
    age_range=(20, 60),
    allowed_les=(3, 5, 7),               # employee / unemployed / inactive deciders
    wage_bounds=(2.0, 170.0),            # employee-decider yivwg bounds
    other_member_income_threshold=50.0,  # |yem|/|yse| above this = earning non-decider
    hours_cap_high=70, hours_floor_low=10, hours_inactive_threshold=5,
    retire_cols=('byr', 'pdi', 'poa', 'psu'),   # CONFIRM FR benefit-receipt columns
)
CONFIG

## 3. Classify households (single / opposite-sex couple)

In [ ]:
ADULT = CONFIG['age_range'][0]
dag = pd.to_numeric(raw['dag'], errors='coerce').fillna(-1)
idp = pd.to_numeric(raw['idpartner'], errors='coerce').fillna(0).astype('int64')
id2partner = dict(zip(raw['idperson'].astype('int64'), idp))
idset = set(raw['idperson'].astype('int64'))

def _mutual(a, b):
    return b != 0 and b in idset and id2partner.get(b, 0) == a

cls = {}
for hh, g in raw.groupby('idhh'):
    ad = g[pd.to_numeric(g['dag'], errors='coerce') >= ADULT]
    n = len(ad)
    if n == 0:
        cls[hh] = 'excl_no_adult'
    elif n == 1:
        cls[hh] = 'single' if int(ad['idpartner'].iloc[0]) == 0 else 'excl_2adult_no_link'
    elif n == 2:
        a, b = ad['idperson'].astype('int64').tolist()
        if _mutual(a, b) and _mutual(b, a):
            gens = sorted(pd.to_numeric(ad['dgn'], errors='coerce').tolist())
            cls[hh] = 'couple_mf' if gens == [0, 1] else 'excl_same_sex'
        else:
            cls[hh] = 'excl_2adult_no_link'
    else:
        cls[hh] = 'excl_3plus_adults'

raw['household_class'] = raw['idhh'].map(cls)
raw['ruro_decider'] = (raw['household_class'].isin(['single', 'couple_mf'])
                       & (pd.to_numeric(raw['dag'], errors='coerce') >= ADULT)).astype(int)
print(raw.groupby('idhh')['household_class'].first().value_counts())
print('\ndeciders flagged:', int(raw['ruro_decider'].sum()))

## 4. The eligibility chain — one step per cell (household funnel printed at each step)

In [ ]:
def funnel(df, label):
    print(f'{label:48s} households={df["idhh"].nunique():6d}  persons={len(df):6d}')
    return df

def keep_all_deciders(df, cond):
    """Keep a household only if EVERY decider satisfies cond."""
    dec = df['ruro_decider'] == 1
    bad = df.loc[dec & ~cond.reindex(df.index), 'idhh']
    return df[~df['idhh'].isin(pd.unique(bad))].copy()

def drop_hh(df, bad_idhh):
    return df[~df['idhh'].isin(pd.unique(bad_idhh))].copy()

# Step 4.0 — baseline: singles + opposite-sex couples only
work = raw[raw['household_class'].isin(['single', 'couple_mf'])].copy()
funnel(work, '4.0 baseline (single + couple_mf)');

In [ ]:
# Step 4.1 — age: every decider in [20, 60]
lo, hi = CONFIG['age_range']
dag = pd.to_numeric(work['dag'], errors='coerce')
work = keep_all_deciders(work, dag.between(lo, hi))
funnel(work, '4.1 age (all deciders 20-60)');

In [ ]:
# Step 4.2 — education: every decider dec == 0 (not currently in education)
if 'dec' in work.columns:
    dec = pd.to_numeric(work['dec'], errors='coerce')
    work = keep_all_deciders(work, dec.eq(0))
funnel(work, '4.2 education (deciders dec==0)');

In [ ]:
# Step 4.3 — retirement/disability: HH sum of (byr+pdi+poa+psu) == 0
# CONFIRM these are the right FR benefit-receipt columns (DE adapter used these four).
rc = [c for c in CONFIG['retire_cols'] if c in work.columns]
print('retire cols used:', rc)
if rc:
    retire = work[rc].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1)
    work['_retire'] = retire
    hh_retire = work.groupby('idhh')['_retire'].sum()
    work = drop_hh(work, hh_retire.index[hh_retire > 0]).drop(columns='_retire')
funnel(work, '4.3 retirement/disability (HH sum == 0)');

In [ ]:
# Step 4.4 — allowed labour status: every decider les in {3, 5, 7}
les = pd.to_numeric(work['les'], errors='coerce')
work = keep_all_deciders(work, les.isin(CONFIG['allowed_les']))
funnel(work, '4.4 allowed LES (deciders in {3,5,7})');

In [ ]:
# Step 4.5 — other household members: drop HH if any NON-decider is
#   working-age-healthy-not-student  OR  earning (|yem| or |yse| > threshold)
lo, hi = CONFIG['age_range']; thr = CONFIG['other_member_income_threshold']
nondec = work['ruro_decider'] == 0
dag = pd.to_numeric(work['dag'], errors='coerce')
ddi = pd.to_numeric(work.get('ddi', 0), errors='coerce').fillna(0)
dec = pd.to_numeric(work.get('dec', 0), errors='coerce').fillna(0)
yem = pd.to_numeric(work.get('yem', 0.0), errors='coerce').fillna(0.0)
yse = pd.to_numeric(work.get('yse', 0.0), errors='coerce').fillna(0.0)
capable = dag.between(lo, hi) & ddi.eq(0) & dec.eq(0)
earning = (yem > thr) | (yse.abs() > thr)
work = drop_hh(work, work.loc[nondec & (capable | earning), 'idhh'])
funnel(work, '4.5 other members (no capable/earning non-deciders)');

In [ ]:
# Step 4.6 — hours cap + inactive transition + wage bounds (employee deciders, les==3)
cap, floor, inact = CONFIG['hours_cap_high'], CONFIG['hours_floor_low'], CONFIG['hours_inactive_threshold']
wlo, whi = CONFIG['wage_bounds']
dec_mask = work['ruro_decider'] == 1
les = pd.to_numeric(work['les'], errors='coerce')
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)
emp = dec_mask & les.eq(3)

n_capped = int((emp & (lhw > cap)).sum())
work.loc[emp & (lhw > cap), 'lhw'] = cap                              # cap >70 to 70
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)
work.loc[emp & (lhw > inact) & (lhw <= floor), 'lhw'] = floor         # raise (5,10] to 10
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)

very_low = emp & (lhw <= inact)                                        # <=5h employees
become_inactive = very_low & les.isin(CONFIG['allowed_les'])
n_inactive = int(become_inactive.sum())
work.loc[become_inactive, 'lhw'] = 0
work.loc[become_inactive, 'les'] = 7
for c in ('yem', 'yse', 'yemse'):
    if c in work.columns:
        work.loc[become_inactive, c] = 0.0
work = drop_hh(work, work.loc[very_low & ~les.isin(CONFIG['allowed_les']), 'idhh'])

# Non-employment labour-status rows must have zero observed hours.
les = pd.to_numeric(work['les'], errors='coerce')
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)
nonemp_hours = dec_mask & les.isin([5, 7]) & (lhw > 0)
n_nonemp_hours_zeroed = int(nonemp_hours.sum())
work.loc[nonemp_hours, 'lhw'] = 0

if 'yivwg' in work.columns:                                            # wage bounds on employees
    dec_mask = work['ruro_decider'] == 1; les = pd.to_numeric(work['les'], errors='coerce')
    yivwg = pd.to_numeric(work['yivwg'], errors='coerce')
    bad_wage = dec_mask & les.eq(3) & yivwg.notna() & ((yivwg < wlo) | (yivwg > whi))
    work = drop_hh(work, work.loc[bad_wage, 'idhh'])

print(f'capped >70h: {n_capped}   ->inactive (<=5h): {n_inactive}   nonemp hours->0: {n_nonemp_hours_zeroed}')
work = work.reset_index(drop=True)
funnel(work, '4.6 hours cap + wage bounds (employees)');

In [ ]:
# The surviving analytical sample, split by household type
singles = work[work['household_class'] == 'single'].reset_index(drop=True)
couples = work[work['household_class'] == 'couple_mf'].reset_index(drop=True)
print('singles households:', singles['idhh'].nunique())
print('couples households:', couples['idhh'].nunique())
work.head()

## 5. Build features (worker flag, hours bands, education, wages) + occupation `loc4`

In [ ]:
df = work.copy()
lhw = pd.to_numeric(df['lhw'], errors='coerce').fillna(0.0)
les = pd.to_numeric(df['les'], errors='coerce')

# FR-SPECIFIC worker flag: mirror scripts/enhanced/enh_RURO_prep.py::_compute_is_worker.
if 'lma' in df.columns:
    lma = pd.to_numeric(df['lma'], errors='coerce').fillna(0)
    use_lma = bool((lma == 1).any() and lma.nunique(dropna=True) > 1)
else:
    lma = None
    use_lma = False

if use_lma:
    df['is_worker'] = ((lma == 1) & (lhw > 0)).astype('int8')
    print('worker rule: lma==1 & lhw>0 (French MNL hierarchy)')
else:
    df['is_worker'] = (les.eq(3) & (lhw > 0)).astype('int8')
    print('worker rule: les==3 & lhw>0 fallback')
df['working'] = (lhw > 0).astype('int8')

# hours bands (certified RURO definitions, identical to DE)
df['working_pt1'] = ((lhw >= 18.5) & (lhw <= 20.5)).astype('int8')
df['working_pt2'] = ((lhw >= 29.5) & (lhw <= 30.5)).astype('int8')
df['working_ft']  = ((lhw >= 37.5) & (lhw <= 40.5)).astype('int8')
df['working_lh']  = ((df['working'] == 1) & (lhw >= 44.5) & (lhw <= 70.0)).astype('int8')

# education (EUROMOD deh): low {0,1,2}, mid {3,4}, high {5}
deh = pd.to_numeric(df['deh'], errors='coerce')
df['educL'] = deh.isin([0, 1, 2]).astype('int8')
df['educM'] = deh.isin([3, 4]).astype('int8')
df['educH'] = deh.eq(5).astype('int8')

# wages: realised wage 0 for non-workers; offer wage kept for everyone
yivwg = pd.to_numeric(df['yivwg'], errors='coerce').fillna(0.0)
df['wage_for_draws'] = yivwg
df['wage_ruro'] = np.where(df['is_worker'].to_numpy() == 1, yivwg.to_numpy(), 0.0)

# age_norm centred on the decider sample mean
dagn = pd.to_numeric(df['dag'], errors='coerce')
mean_age = float(dagn[df['ruro_decider'] == 1].mean())
df['age_norm'] = dagn - mean_age
df['age_norm2'] = df['age_norm'] ** 2
df['female'] = (pd.to_numeric(df['dgn'], errors='coerce') == 0).astype('int8')

# FR-SPECIFIC (region): unlike DE (constant 0, dropped), FR drgn1/drgur/drgmd/drgru VARY -> KEEP.
for c in ['drgn1', 'drgur', 'drgmd', 'drgru']:
    print(f'  region {c}: {"present (keep for FR)" if c in df.columns else "absent"}')
# FR-SPECIFIC (earnings): FR splits employment income into yem00 / yemxp (35h overtime).
#   DE used a single yem. Relevant when you mutate earnings for pricing alternatives.
print('  yem00/yemxp present:', ('yem00' in df.columns, 'yemxp' in df.columns))

df[['idhh','dag','dgn','les','lhw','is_worker','working_ft','working_lh','educL','educM','educH','wage_ruro']].head(10)

In [ ]:
# Dedicated loc_ruro / loc4 construction (certified task groups)
from dclaborsupply_app.de.data_prep import collapse_loc_to_loc4

if 'loc' not in df.columns:
    raise KeyError("Cannot build loc4: raw occupation column 'loc' is missing.")

if 'loc_raw' not in df.columns:
    df['loc_raw'] = df['loc']

loc_src = pd.to_numeric(df['loc'], errors='coerce').fillna(-2).astype('int16')
isw = pd.to_numeric(df['is_worker'], errors='coerce').fillna(0).astype(int)
df['loc_ruro'] = loc_src
valid_worker_loc = loc_src.isin(list(range(0, 10)))
df.loc[(isw == 1) & ~valid_worker_loc, 'loc_ruro'] = -2
df.loc[isw != 1, 'loc_ruro'] = -1

loc4, loc_armed = collapse_loc_to_loc4(df['loc_ruro'])
df['loc4'] = loc4.astype('int16')
df['loc_armed'] = loc_armed.astype('int8')

model_worker = isw == 1
model_nonworker = ~model_worker
unknown_worker = model_worker & (pd.to_numeric(df['loc4'], errors='coerce') == -2)
bad_worker = model_worker & ~pd.to_numeric(df['loc4'], errors='coerce').isin([-2, 1, 2, 3, 4])
bad_nonworker = model_nonworker & (pd.to_numeric(df['loc4'], errors='coerce') != -1)
assert not bad_worker.any(), f"is_worker rows with unsupported loc4 remain: {int(bad_worker.sum())}"
assert not bad_nonworker.any(), f"non-is_worker rows with loc4 != -1 remain: {int(bad_nonworker.sum())}"
if unknown_worker.any():
    bad_cols = [c for c in ['idhh', 'idperson', 'les', 'lhw', 'is_worker', 'loc', 'loc_ruro', 'loc4', 'loc_armed'] if c in df.columns]
    print(
        f"Observed is_worker rows with loc4=-2: {int(unknown_worker.sum())} "
        f"({df.loc[unknown_worker, 'idhh'].nunique()} households). Keeping them: "
        "MNL convention treats -2 as unknown observed occupation; simulated working draws are mode-imputed to {1,2,3,4}."
    )
    display(df.loc[unknown_worker, bad_cols].head(20))

print('loc4 counts:')
display(df['loc4'].value_counts(dropna=False).sort_index())
df[['idhh', 'idperson', 'les', 'lhw', 'is_worker', 'loc', 'loc_ruro', 'loc4', 'loc_armed']].head(10)

## 6. Price the observed state through EUROMOD (sanity)

**Decision — bsa00yn_a=1 full-entitlement pricing.** EUROMOD is priced with social-assistance
take-up forced on, so every alternative carries the *entitlement*; the household-specific take-up
mask is applied later at assembly (§11). This keeps pricing deterministic and separates the tax-
benefit calculation from the behavioural take-up model. Guarded by `SKIP_PRICING`.

In [ ]:
# connector + EUROMOD-input frame (built regardless); the actual run is guarded.
from dclaborsupply_app.euromod import EuromodConnector
MODEL_ROOT = Path(r'C:\Users\hisham\MNL\EUROMOD-STORAGE\Euromod_model\EUROMOD_RELEASES_J2.0+')
FR_COUNTRY, FR_SYSTEM, FR_DATASET = 'FR', 'FR_2015', 'FR_2016_a3'
conn = EuromodConnector(str(MODEL_ROOT))

derived = {'household_class','ruro_decider','is_worker','working','working_pt1','working_pt2',
           'working_ft','working_lh','educL','educM','educH','wage_for_draws','wage_ruro',
           'age_norm','age_norm2','female','loc_raw','loc_ruro','loc4','loc_armed'}
em_input = df[[c for c in df.columns if c not in derived]].copy()
print('EUROMOD input columns:', em_input.shape[1])

if not SKIP_PRICING:
    res = conn.run(em_input, country=FR_COUNTRY, system=FR_SYSTEM, dataset=FR_DATASET)
    priced = res.output
    print('priced shape:', priced.shape, '| ils_dispy present:', 'ils_dispy' in priced.columns)
else:
    print('SKIP_PRICING=True: observed-state EUROMOD run skipped (em_input + conn built).')

In [ ]:
# disposable-income sanity (only if the observed run was executed)
if 'priced' in globals() and 'ils_dispy' in priced.columns:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    c = pd.to_numeric(priced['ils_dispy'], errors='coerce')
    print(c.describe())
    plt.figure(figsize=(7,4)); plt.hist(c.clip(lower=0), bins=40)
    plt.title('FR disposable income (ils_dispy)'); plt.xlabel('EUR/month'); plt.ylabel('count')
    plt.tight_layout(); plt.show()
else:
    print('SKIP_PRICING: observed ils_dispy histogram skipped.')

## 7. Restrict to singles (the modelled scope)

In [ ]:
# ── 8. Restrict to singles (the trial scope) ──────────────────────────────
singles_df = df[df['household_class'] == 'single'].reset_index(drop=True)
print('singles households:', singles_df['idhh'].nunique(), '| persons:', len(singles_df))

# the modelled individuals = the single deciders (one per single household)
singles_dec = singles_df[singles_df['ruro_decider'] == 1].reset_index(drop=True)
print('single deciders   :', len(singles_dec),
      '| female share   :', round(singles_dec['female'].mean(), 3))

singles_dec[['idhh','dag','dgn','female','les','lhw','is_worker',
             'wage_ruro','educL','educM','educH']].head(10)

# educ3 for gsur/occ strata (0=low,1=mid,2=high)
if 'educ3' not in singles_dec.columns:
    singles_dec['educ3'] = np.select(
        [singles_dec['educL']==1, singles_dec['educM']==1, singles_dec['educH']==1],
        [0, 1, 2], default=-1).astype(int)
singles_df = singles_df.copy()

## 8. Fold the certified feature constructions onto `singles_dec` — gsur + pexp

One cell replaces the v3/v4/v5 engine-ready archaeology. Two certified constructions:
- **gsur** (group unemployment, the exclusion-restriction covariate): merged from the GSURv2
  Stage-A lookup on `(drgn1, educ3, dgn→sex)`, opportunity year 2015 for the 2016 wave.
- **pexp_years** (potential experience): certified priority `liwwh/12 → (year−dew) → (dag−6−dey)`,
  capped at `dag−15`, then the harmonise `/20` rescale (raw kept for the wage-draw Mincer mean).
Region dummies and `age_norm` are built downstream by the loader / assembly, not here.

In [ ]:
# --- gsur onto singles_dec (certified GSURv2 Stage-A lookup, y2015 opportunity year) ---
GSUR_LKP = Path(r'C:\Users\hisham\MNL\EUROMOD-STORAGE\Data\external\FR_gsur_ruro_v2_stageA_y2015.parquet')
assert GSUR_LKP.is_file(), f'gsur lookup not found: {GSUR_LKP}'
lkp = pd.read_parquet(GSUR_LKP)
lkp = lkp.loc[lkp['gsur'].notna(), ['drgn1','educ3','sex','gsur']]
assert len(lkp) == 48 and lkp['gsur'].between(0, 1).all()
_lo, _hi = float(lkp['gsur'].min()), float(lkp['gsur'].max())
_keys = pd.DataFrame({'drgn1': pd.to_numeric(singles_dec['drgn1'], errors='coerce').astype('int64'),
                      'educ3': pd.to_numeric(singles_dec['educ3'], errors='coerce').astype('int64'),
                      'sex':   pd.to_numeric(singles_dec['dgn'], errors='coerce').map({1:'M',0:'F'})})
assert _keys.notna().all().all(), 'unmapped dgn / non-numeric drgn1|educ3'
_mm = _keys.merge(lkp, on=['drgn1','educ3','sex'], how='left', validate='many_to_one')
assert _mm['gsur'].notna().all(), 'gsur match rate < 100%'
singles_dec = singles_dec.assign(gsur=_mm['gsur'].to_numpy())
print(f'gsur merged: n={len(singles_dec)} range [{_lo:.4f},{_hi:.4f}]')

# --- pexp_years (certified liwwh->dew->dey, cap dag-15, /20 rescale) ---
_dag  = pd.to_numeric(singles_dec['dag'], errors='coerce')
_year = pd.Series(2016.0, index=singles_dec.index)
_p_liwwh = pd.Series(np.nan, index=singles_dec.index)
if 'liwwh' in singles_dec:
    _liwwh = pd.to_numeric(singles_dec['liwwh'], errors='coerce')
    _p_liwwh = (_liwwh/12.0).where(_liwwh.notna() & (_liwwh>0), np.nan)
_p_dew = pd.Series(np.nan, index=singles_dec.index)
if 'dew' in singles_dec:
    _dew = pd.to_numeric(singles_dec['dew'], errors='coerce')
    _ok = _dew.notna() & (_dew!=-1) & (_dew>=1900) & (_dew<=2100) & (_dew<=_year)
    _p_dew = (_year-_dew).where(_ok, np.nan)
_p_dey = pd.Series(np.nan, index=singles_dec.index)
if 'dey' in singles_dec:
    _dey = pd.to_numeric(singles_dec['dey'], errors='coerce')
    _ok = _dey.notna() & (_dey>=0) & (_dey<=100)
    _p_dey = (_dag-6.0-_dey).where(_ok, np.nan)
_pexp_raw = _p_liwwh.where(_p_liwwh.notna(), _p_dew).where(lambda s: s.notna(), _p_dey)
_cap = (_dag-15.0).clip(lower=0.0)
singles_dec['pexp_years_raw'] = np.minimum(_pexp_raw.fillna(0.0).clip(lower=0.0), _cap.fillna(0.0)).astype(float)
singles_dec['pexp_years']  = singles_dec['pexp_years_raw']/20.0
singles_dec['pexp_years2'] = singles_dec['pexp_years']**2
assert singles_dec['pexp_years'].between(0, 2.6).all()
print('pexp_years: med=%.3f max=%.3f' % (singles_dec['pexp_years'].median(), singles_dec['pexp_years'].max()))

## 9. Certified B-pool draws

**Decision — certified draws, not the uniform proposal.** The uniform-hours / occ-fixed draws used
in the trial could not form the observed 35h reference-band atom and left the occupation block
unidentified (occ_spec='fixed' imputed every simulated working draw to one loc4 → a ridge, ridge
probe ≈ −0.0002). The certified B-pool design fixes both: a **D1 five-mode hours mixture** with a
peaked reference band, **empirical occupation sampling** by (dgn,educ3) stratum, and **W1
occupation-conditional log-normal wages** — with matching `log_q_*` so the importance weights stay
exact. `log_prior = log_q_E + working·(log_q_H+log_q_W+log_q_Occ)`; pi0=0.10; 100 sim + 1 observed;
master seed 2026 with per-household child seeds (mirrors `build_bpool_singles.py`).

In [ ]:
# ── P2a-0. Certified draw modules: import + signatures + 5-person smoke ─────
import sys, inspect, importlib, json
from pathlib import Path
import numpy as np

# locate MNL repo root (works whether the kernel cwd is repo root or the notebook dir)
MNL_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'scripts/bpool/hours_mixture_d1.py').exists()), None)
if MNL_ROOT is None:
    raise FileNotFoundError('Could not locate MNL repo root (scripts/bpool/hours_mixture_d1.py).')
# the three modules import each other by BARE name (see build_bpool_singles.py) -> put
# their dirs on sys.path and import by bare name, exactly as the production builder does.
for sub in ('scripts/bpool', 'scripts/pilot'):
    d = str(MNL_ROOT / sub)
    if d not in sys.path:
        sys.path.insert(0, d)

mods = {n: importlib.import_module(n)
        for n in ['hours_mixture_d1', 'occ_draw_empirical', 'pilot_wage_draw']}
for name, m in mods.items():
    print(f'== {name} -> {m.__file__}')
    for fn, o in vars(m).items():
        if inspect.isfunction(o) and not fn.startswith('_'):
            print(f'   {fn}{inspect.signature(o)}')
print('missing: none — all three importable')

MINCER = MNL_ROOT / 'scripts/pilot/config/pilot_mincer_coefficients_v1.json'  # note: config/ subdir
assert MINCER.is_file(), f'mincer json not found: {MINCER}'
mc = json.load(open(MINCER))
print('mincer wage_model_W1 keys:', list(mc['wage_model_W1'].keys()),
      '| sigma:', round(mc['wage_model_W1']['sigma'], 6))

# ── 5-person smoke: hours (D1) → loc4 (empirical) → wage (W1), with lockstep log_q ──
rng = np.random.default_rng(0)
n = 5
dgn   = np.array([1, 1, 0, 0, 1]); educ3 = np.array([0, 1, 2, 0, 2])
hours, lq_h = mods['hours_mixture_d1'].draw_hours_d1(n, rng)
loc4,  lq_o = mods['occ_draw_empirical'].draw_loc4(dgn, educ3, rng)
wres = mods['pilot_wage_draw'].draw_pilot_wages(
    n=n, educL=(educ3 == 0).astype(float), educH=(educ3 == 2).astype(float),
    pexp_years=np.full(n, 15.0), pexp_years2=np.full(n, 225.0),
    loc4=loc4, year_tag=np.full(n, 2), mincer_payload=mc, seed=0, draw_method='halton')
print('\n5-person working smoke:')
for i in range(n):
    print(f'  dgn={dgn[i]} educ3={educ3[i]} | hours={hours[i]:5.1f} loc4={loc4[i]} '
          f'wage={wres["wage"][i]:6.2f} | log_q_H={lq_h[i]:+.3f} log_q_Occ={lq_o[i]:+.3f} '
          f'log_q_W={wres["log_q_wage"][i]:+.3f}')
# log_prior for a WORKING alt = log(1-pi0) + log_q_H + log_q_W + log_q_Occ
lp = np.log(0.9) + lq_h + lq_o + np.asarray(wres['log_q_wage'])
print('  log_prior (working) =', np.round(lp, 3), '| wage draw method:', wres['method'])


In [ ]:
# ── P2a-1. Rebuild singles draws under the CERTIFIED B-pool design ──────────
# Mirrors scripts/bpool/build_bpool_singles.py (draw loop) + its three modules.
# Produces draws_p2a (n_hh * 101 rows) with the certified log_q design on singles_dec.
import sys, json
from pathlib import Path
import numpy as np, pandas as pd

MNL_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'scripts/bpool/build_bpool_singles.py').exists()), None)
assert MNL_ROOT, 'MNL repo root not found (scripts/bpool/build_bpool_singles.py)'
for sub in ('scripts/bpool', 'scripts/pilot'):
    d = str(MNL_ROOT / sub)
    if d not in sys.path:
        sys.path.insert(0, d)
import hours_mixture_d1 as HM      # D1 six-uniform mixture   -> draw_hours_d1  (imported build_bpool_singles.py:52)
import occ_draw_empirical as OE    # empirical (dgn,educ3) loc4 -> draw_loc4, FREQ_TABLE      (:53)
import pilot_wage_draw as PW       # W1 occ-conditional lognormal -> draw_pilot_wages          (:54)
MINCER = MNL_ROOT / 'scripts/pilot/config/pilot_mincer_coefficients_v1.json'  # build_bpool_singles.py:67-69
mc = json.load(open(MINCER))
sigma = float(mc['wage_model_W1']['sigma'])

# constants mirror build_bpool_singles.py:59-62
N_DRAWS, PI0 = 100, 0.10
LOG_PI0, LOG_1_PI0 = np.log(PI0), np.log(1.0 - PI0)
YEAR_TAG = 2   # year_tag absent -> 2016 reference (1=2015,2=2016,3=2017); wage shifts only for ==1/==3

req = ['idhh', 'idperson', 'dgn', 'educ3', 'educL', 'educH',
       'pexp_years_raw', 'lhw', 'yivwg', 'loc4', 'is_worker']
miss = [c for c in req if c not in singles_dec.columns]
assert not miss, f'singles_dec missing required columns: {miss}'

carry_cols = [c for c in singles_dec.columns if c != 'loc4']   # loc4 is set per-alt (drawn)

# per-household child seeds — EXACT mirror of build_bpool_singles.py:307-308
ids = singles_dec['idhh'].to_numpy()
_seed_rng = np.random.default_rng(2026)                                   # :307  default_rng(seed=2026)
hh_seeds = {int(uid): int(_seed_rng.integers(0, 2**31)) for uid in ids}   # :308  per-uid child seed

def _band_flags(h, wk):   # mirror build_bpool_singles.py:216-219 (F35 = reference, no flag)
    return (float((36.5 <= h <= 40.5) and wk), float((17.5 <= h < 21.5) and wk),
            float((28.5 <= h < 30.5) and wk), float((44.5 <= h <= 70.0) and wk))

blocks = []
for r in singles_dec.to_dict('records'):
    uid = int(r['idhh'])
    hh_rng = np.random.default_rng(hh_seeds[uid])   # :315  per-HH PCG64 (employment/occ/hours)
    seed = hh_seeds[uid]                             # :316  wage Halton seed
    dgn, educ3 = int(r['dgn']), int(r['educ3'])
    educL, educH = float(r['educL']), float(r['educH'])
    pexp = float(r['pexp_years_raw']); pexp2 = pexp * pexp

    # employment  (mirror :119-123)
    u = hh_rng.uniform(size=N_DRAWS)
    working = (u >= PI0).astype(np.int8); nwk = int(working.sum())
    lqE = np.where(working == 1, LOG_1_PI0, LOG_PI0)
    # occupation (mirror :128-136)
    loc4 = np.full(N_DRAWS, -1, dtype=np.int32); lqO = np.zeros(N_DRAWS)
    # hours      (mirror :141-147)
    hours = np.zeros(N_DRAWS); lqH = np.zeros(N_DRAWS)
    # wage       (mirror :152-170)
    wage = np.zeros(N_DRAWS); lqW = np.zeros(N_DRAWS); wmu = np.full(N_DRAWS, np.nan)
    if nwk > 0:
        l4, lo = OE.draw_loc4(np.full(nwk, dgn), np.full(nwk, educ3), hh_rng)
        loc4[working == 1] = l4; lqO[working == 1] = lo
        h, lh = HM.draw_hours_d1(nwk, hh_rng)
        hours[working == 1] = h; lqH[working == 1] = lh
        res = PW.draw_pilot_wages(n=nwk, educL=np.full(nwk, educL), educH=np.full(nwk, educH),
                                  pexp_years=np.full(nwk, pexp), pexp_years2=np.full(nwk, pexp2),
                                  loc4=loc4[working == 1], year_tag=np.full(nwk, YEAR_TAG),
                                  mincer_payload=mc, seed=seed, draw_method='halton')
        wage[working == 1] = res['wage']; lqW[working == 1] = res['log_q_wage']; wmu[working == 1] = res['mu']
    # log_prior: build_bpool_singles.py:175-177 sums (Occ+H+W); we sum in the LOADER's order
    # (H+W+Occ, engine_ready.py:289) so the wave01 identity holds bit-exactly downstream.
    lp = lqE + working.astype(np.float64) * (lqH + lqW + lqO)

    sim = pd.DataFrame({c: [r[c]] * N_DRAWS for c in carry_cols})
    sim['draw'] = np.arange(1, N_DRAWS + 1); sim['is_chosen'] = np.int8(0)
    sim['working'] = working.astype(float); sim['hours'] = hours; sim['wage'] = wage; sim['loc4'] = loc4.astype(float)
    sim['log_q_E'] = lqE; sim['log_q_H'] = lqH; sim['log_q_W'] = lqW; sim['log_q_Occ'] = lqO
    sim['log_prior'] = lp; sim['wage_mu'] = wmu
    hh = sim['hours'].to_numpy(); wkm = (sim['working'] == 1).to_numpy()
    sim['working_ft']  = ((hh >= 36.5) & (hh <= 40.5) & wkm).astype(float)
    sim['working_pt1'] = ((hh >= 17.5) & (hh <  21.5) & wkm).astype(float)
    sim['working_pt2'] = ((hh >= 28.5) & (hh <  30.5) & wkm).astype(float)
    sim['working_lh']  = ((hh >= 44.5) & (hh <= 70.0) & wkm).astype(float)

    # chosen row (draw==0): observed values preserved, all log_q=0, log_prior=0 (IS anchor; :224-267)
    ch = pd.DataFrame({c: [r[c]] for c in carry_cols})
    ch_h = float(r['lhw']); ch_wk = int(r['is_worker']) == 1
    ch['draw'] = 0; ch['is_chosen'] = np.int8(1)
    ch['working'] = float(int(r['is_worker'])); ch['hours'] = ch_h; ch['wage'] = float(r['yivwg']); ch['loc4'] = float(r['loc4'])
    ch['log_q_E'] = ch['log_q_H'] = ch['log_q_W'] = ch['log_q_Occ'] = ch['log_prior'] = 0.0
    ch['wage_mu'] = np.nan
    ft, pt1, pt2, lhf = _band_flags(ch_h, ch_wk)
    ch['working_ft'], ch['working_pt1'], ch['working_pt2'], ch['working_lh'] = ft, pt1, pt2, lhf
    blocks.append(pd.concat([ch, sim], ignore_index=True))

draws_p2a = pd.concat(blocks, ignore_index=True)
draws_p2a['idhh_true'] = draws_p2a['idhh']; draws_p2a['idperson_true'] = draws_p2a['idperson']
if 'educM' not in draws_p2a.columns:
    draws_p2a['educM'] = (pd.to_numeric(draws_p2a['educ3'], errors='coerce') == 1).astype(float)
draws_p2a['prior'] = np.clip(np.exp(np.clip(draws_p2a['log_prior'], -700, 700)), 1e-16, None)  # wave01 canonical prior

# ────────────────────────── GATES ──────────────────────────
d = draws_p2a; n_hh = singles_dec['idhh'].nunique()
assert len(d) == n_hh * 101, (len(d), n_hh * 101)
assert (d.groupby('idhh')['draw'].size() == 101).all(), '101 alts/person'
ch = d[d['draw'] == 0]
assert (ch['log_prior'] == 0).all() and (ch['is_chosen'] == 1).all()
obs = singles_dec.set_index('idhh')
chi = ch.set_index('idhh')
assert np.allclose(chi['hours'], pd.to_numeric(obs['lhw']).reindex(chi.index)), 'chosen hours preserved'
assert np.allclose(chi['wage'],  pd.to_numeric(obs['yivwg']).reindex(chi.index)), 'chosen wage preserved'
sne = d[(d['draw'] > 0) & (d['working'] == 0)]
assert (sne['hours'] == 0).all() and (sne['wage'] == 0).all() and (sne['loc4'] == -1).all()
assert np.allclose(sne['log_prior'], np.log(0.1)), 'non-emp log_prior == log(0.1)'
atoms = np.log(HM.DEFAULT_WEIGHTS / HM.BAND_WIDTHS)               # 6 mixture atoms from the module
sw = d[(d['draw'] > 0) & (d['working'] == 1)]
lqh = sw['log_q_H'].to_numpy()
assert np.min(np.abs(lqh[:, None] - atoms[None, :]), axis=1).max() < 1e-9, 'log_q_H not in the 6 atoms'
dev = (d['log_prior'] - (d['log_q_E'] + d['working'] * (d['log_q_H'] + d['log_q_W'] + d['log_q_Occ']))).abs().max()
assert dev == 0.0, f'wave01 identity dev={dev}'
# occ shares per (dgn,educ3) vs module FREQ_TABLE (±2%)
occ_ok = True
for (dg, ed), grp in sw.groupby(['dgn', 'educ3']):
    emp = grp['loc4'].value_counts(normalize=True).reindex([1, 2, 3, 4]).fillna(0).to_numpy()
    tab = OE.FREQ_TABLE[(int(dg), int(ed))]; tab = tab / tab.sum()
    md = float(np.abs(emp - tab).max()); occ_ok &= md < 0.02
    print(f'  occ (dgn={dg},educ3={ed}) max|emp-table|={md:.4f}  n={len(grp)}')
assert occ_ok, 'occ share drift > 2% in some stratum'
# wage: residual log(w)-mu ~ N(0, sigma)
resid = np.log(sw['wage'].to_numpy()) - sw['wage_mu'].to_numpy()
print(f'\nwage residual (log w - mu): mean={resid.mean():+.4f} sd={resid.std():.4f}  (sigma={sigma:.4f})')
for l4 in (1, 2, 3, 4):
    rl = np.log(sw.loc[sw['loc4'] == l4, 'wage']) - sw.loc[sw['loc4'] == l4, 'wage_mu']
    print(f'  loc4={l4}: n={len(rl)} resid mean={rl.mean():+.4f} sd={rl.std():.4f}')
assert abs(resid.mean()) < 0.05 and abs(resid.std() - sigma) < 0.05, 'wage residual not ~N(0,sigma)'

print(f'\ndraws_p2a: {draws_p2a.shape}  ({n_hh} HH x 101)  — ALL GATES PASS')
print('cols:', sorted(draws_p2a.columns.tolist()))


In [ ]:
# === OPERATOR-GATED PRODUCTION EXCEPTION: freeze the draws_p2a geometry ========
# The SOLE authorized v2 write outside NOTEBOOK_OUTPUT_ROOT (banner + addendum s15):
# an immutable, hash-declared freeze of the in-memory draws_p2a frame to
# region_live_v1/inputs/ -- the G-0 frozen input of the production Phase 1-2 runner
# (manager decision D-1 v2). It freezes the ALREADY-BUILT draws_p2a only: no draw
# generation, no engine-ready derivation, no other production artifact, and it never
# rewrites an existing valid freeze.
if EXPORT_PRODUCTION_GEOMETRY:
    import hashlib as _hashlib, json as _json, os as _os, yaml as _yaml
    from datetime import datetime as _dt, timezone as _tz
    from pathlib import Path as _P

    assert 'draws_p2a' in globals(), 'draws_p2a must exist (run the P2a-1 draws cell first)'

    _mnl = next((p for p in [_P.cwd(), *_P.cwd().parents]
                 if (p / 'scripts/p2a/configs/p2a_regionlive_rebuild_v1.yaml').exists()), None)
    assert _mnl is not None, 'MNL repo root with scripts/p2a/configs not found'
    _cfg = _yaml.safe_load((_mnl / 'scripts/p2a/configs/p2a_regionlive_rebuild_v1.yaml')
                           .read_text(encoding='utf-8'))
    _gc = _cfg['frozen_inputs']['draws_geometry']

    # seed verification (contract seed 2026): the child-seed dict must reproduce from
    # the master seed. Verification only -- no draw-generation function is invoked.
    assert int(_gc['expected_seed']) == 2026, 'config expected_seed != 2026'
    if 'hh_seeds' in globals() and 'singles_dec' in globals():
        _chk_rng = np.random.default_rng(2026)
        _chk = {int(_u): int(_chk_rng.integers(0, 2**31))
                for _u in singles_dec['idhh'].to_numpy()}
        assert _chk == hh_seeds, 'hh_seeds do not reproduce from master seed 2026'

    # complete in-memory column set (no column selection); canonical stable sort
    _geo = draws_p2a.sort_values(['idhh', 'draw'], kind='mergesort').reset_index(drop=True)

    def _contract(_d):
        assert len(_d) == int(_gc['expected_rows']), f"rows {len(_d)} != {_gc['expected_rows']}"
        assert _d['idhh'].nunique() == int(_gc['expected_hh']), 'household count != 1555'
        assert (_d.groupby('idhh')['draw'].size() == int(_gc['expected_alts_per_hh'])).all(), \
            'not exactly 101 alternatives per household'
        _missing = [c for c in _gc['required_columns'] if c not in _d.columns]
        assert not _missing, f'required geometry columns missing: {_missing}'
        assert not _d.duplicated(subset=['idhh', 'draw']).any(), 'duplicate (idhh, draw)'
        _ch = _d[_d['draw'] == 0]
        assert len(_ch) == int(_gc['expected_hh']) and _ch['idhh'].is_unique, \
            'draw==0 rows are not exactly one per household'
        assert (pd.to_numeric(_ch['is_chosen']) == 1).all(), 'chosen rows must have is_chosen==1'
        assert (_ch['log_prior'] == 0).all(), 'chosen rows must have log_prior==0'
        assert (_d['prior'] > 0).all(), 'prior must be strictly positive'
        assert np.isfinite(_d['log_prior'].to_numpy(dtype='float64')).all(), 'log_prior not finite'
        _dev = (_d['log_prior'] - (_d['log_q_E'] + _d['working']
                * (_d['log_q_H'] + _d['log_q_W'] + _d['log_q_Occ']))).abs().max()
        assert _dev == 0.0, f'wave01 identity dev={_dev}'   # exact equality, as the draws gate

    _contract(_geo)

    _dst = _mnl / 'outputs/p2a_singles2016/region_live_v1/inputs'
    assert _dst.resolve().as_posix().endswith('outputs/p2a_singles2016/region_live_v1/inputs'), \
        'freeze may write ONLY under region_live_v1/inputs'
    _pq = _dst / 'fr_p2a_draws_geometry__singles.parquet'
    _mt = _dst / 'fr_p2a_draws_geometry__meta.json'

    def _sha256(_p):
        _h = _hashlib.sha256()
        with open(_p, 'rb') as _f:
            for _b in iter(lambda: _f.read(1 << 20), b''):
                _h.update(_b)
        return _h.hexdigest()

    if _pq.exists() and _mt.exists():
        _old = _json.loads(_mt.read_text(encoding='utf-8'))
        assert _old.get('sha256') == _sha256(_pq), \
            'existing geometry parquet does not match its metadata sha256 -- refusing to overwrite'
        _contract(pd.read_parquet(_pq))
        print(f'geometry already frozen and contract-valid -- not rewritten: {_pq}')
        print(f"  sha256={_old['sha256']}  created_at_utc={_old.get('created_at_utc')}")
    elif _pq.exists() != _mt.exists():
        raise RuntimeError('inconsistent freeze state: exactly one of parquet/meta exists '
                           '-- refusing to overwrite either')
    else:
        _dst.mkdir(parents=True, exist_ok=True)
        _tmp = _dst / 'fr_p2a_draws_geometry__singles.parquet.tmp'
        _geo.to_parquet(_tmp)
        _sha = _sha256(_tmp)                      # hash the completed temporary file
        _contract(pd.read_parquet(_tmp))          # re-validate the on-disk bytes
        _os.replace(_tmp, _pq)                    # atomic promotion after all checks pass
        _meta = {
            'status': 'frozen_production_input',
            'produced_by': 'notebooks/fr_singles_pipeline_v2.ipynb',
            'seed': 2026,
            'draw_design': _gc['draw_design'],
            'sha256': _sha,
            'n_rows': int(len(_geo)),
            'n_households': int(_geo['idhh'].nunique()),
            'alternatives_per_household': int(_gc['expected_alts_per_hh']),
            'n_columns': int(_geo.shape[1]),
            'columns': [str(c) for c in _geo.columns],
            'dtypes': {str(c): str(t) for c, t in _geo.dtypes.items()},
            'required_columns': [str(c) for c in _gc['required_columns']],
            'created_at_utc': _dt.now(_tz.utc).isoformat(),
            'manager_decisions':
                'docs/France_case/P2a/FR_P2a_region_live_manager_decisions_v2.md',
            'production_plan':
                'docs/France_case/P2a/FR_P2a_region_live_production_rebuild_plan_v2.md',
        }
        _mtmp = _dst / 'fr_p2a_draws_geometry__meta.json.tmp'
        _mtmp.write_text(_json.dumps(_meta, indent=2), encoding='utf-8')
        _os.replace(_mtmp, _mt)
        print(f'production geometry FROZEN: {_pq}')
        print(f'  sha256={_sha}  rows={len(_geo)}  cols={_geo.shape[1]}')
else:
    print('[SKIPPED: EXPORT_PRODUCTION_GEOMETRY=False] production geometry freeze not run.')


## 10. Deterministic full pricing of the draws (bsa00yn_a=1), chunked + resumable

In [ ]:
# pricing configuration (FR earnings policy + full-entitlement baseline + runner)
from dclaborsupply_app.euromod import EuromodPricingRunner
from typing import Any, Mapping, Dict
WEEKS_PER_MONTH = 13.0/3.0

def fr_earnings_policy(member: Mapping[str, Any], *, hours: float, wage: float,
                       weeks_per_month: float = WEEKS_PER_MONTH) -> Dict[str, float]:
    # split employment income at the 35h French workweek: yem00 std, yemxp overtime.
    try: yse = float(member.get('yse', 0.0) or 0.0)
    except (TypeError, ValueError): yse = 0.0
    h, wrate, weeks = float(hours), float(wage), float(weeks_per_month)
    std = wrate*min(h,35.0)*weeks; ot = wrate*max(h-35.0,0.0)*weeks; yem = std+ot
    out = {'lhw': h, 'yem00': std, 'yemxp': ot, 'yem': yem}
    if 'yemse' in member: out['yemse'] = yem + yse
    return out

single_idhh = df.loc[df['household_class']=='single', 'idhh'].unique()
_keep = list(em_input.columns) + ['ruro_decider']
baseline = df.loc[df['idhh'].isin(single_idhh), _keep].copy()
baseline_full = baseline.copy(); baseline_full['bsa00yn_a'] = 1     # full entitlement; mask at assembly
WORKING_ZERO_COLS = tuple(c for c in ('bun','bunmt','bunct','bsa') if c in baseline_full.columns)
FULLYEAR_COLS = {c: v for c, v in {'yemmy':12.0,'lunmy':0.0}.items() if c in baseline_full.columns}

def fr_earnings_policy_v2(member, *, hours, wage, weeks_per_month=WEEKS_PER_MONTH):
    ov = fr_earnings_policy(member, hours=hours, wage=wage, weeks_per_month=weeks_per_month)
    if float(hours) > 0:
        for c in WORKING_ZERO_COLS: ov[c] = 0.0
        ov.update(FULLYEAR_COLS)
    return ov

KEEP = ['idhh','idperson','source_idhh','source_idorighh','source_idperson',
        'ruro_decider','dgn','draw','ils_dispy','bsa00_s']
runner_full = EuromodPricingRunner(conn, fr_earnings_policy_v2)
hh_all = sorted(single_idhh); CHUNK = 200
print('pricing setup ready | working-zero cols:', WORKING_ZERO_COLS, '| fullyear:', FULLYEAR_COLS)

In [ ]:
# Full pricing of the certified draws. Resumable: cached chunks are loaded; with
# SKIP_PRICING no EUROMOD call is made (all chunks must already be cached).
PRICE_CACHE_SHARED = Path('fr_singles_pricing_p2a')          # git-tracked shared read cache (input)
PRICE_DIR2 = NOTEBOOK_OUTPUT_ROOT / 'fr_singles_pricing_p2a' # v2-generated chunks isolated here
PRICE_DIR2.mkdir(parents=True, exist_ok=True)
if RUN_PRICING:
    print('[pricing] RUN_PRICING=True -> EUROMOD will price any chunk missing from the caches.')
else:
    print('[SKIPPED: RUN_PRICING=False] EUROMOD pricing off; loading cached chunks only '
          '(shared read cache + any v2 dev cache).')
alt2 = draws_p2a.copy()
alt2['source_idhh']      = alt2['idhh_true']
alt2['decider_idperson'] = alt2['idperson_true']

outs, _t_start = [], time.time()
for i in range(0, len(hh_all), CHUNK):
    part = hh_all[i:i+CHUNK]
    f_dev = PRICE_DIR2 / f'priced_{i:05d}.parquet'
    f_shared = PRICE_CACHE_SHARED / f'priced_{i:05d}.parquet'
    if f_dev.exists():    outs.append(pd.read_parquet(f_dev)); continue
    if f_shared.exists(): outs.append(pd.read_parquet(f_shared)); continue
    if not RUN_PRICING:
        raise RuntimeError(f'pricing chunk {i:05d} missing and RUN_PRICING=False (no EUROMOD call)')
    r = runner_full.price(alt2[alt2['source_idhh'].isin(part)],
                          baseline_full[baseline_full['idhh'].isin(part)],
                          country=FR_COUNTRY, system=FR_SYSTEM, dataset=FR_DATASET,
                          alt_key_cols=['draw'], weeks_per_month=WEEKS_PER_MONTH)
    bad = [e for e in r.errors if 'uprate' not in e.lower()]
    assert not bad, bad[:3]
    o = r.output[[c for c in KEEP if c in r.output.columns]]
    o.to_parquet(f_dev); outs.append(o)
    print(f'chunk {i//CHUNK+1}  rows={len(o)}  {time.time()-_t_start:.0f}s')
priced_p2a = pd.concat(outs, ignore_index=True)
print('priced:', priced_p2a.shape, '| hh:', priced_p2a['source_idhh'].nunique())

# determinism gate (same as 16-gate, on the new asset)
d = (priced_p2a[priced_p2a['ruro_decider']==1]
     .merge(alt2[['source_idhh','draw','hours','wage']], on=['source_idhh','draw']))
d['hw'] = d['hours'].round(6).astype(str)+'_'+d['wage'].round(6).astype(str)
g = d.groupby(['source_idperson','hw'])['ils_dispy'].nunique()
assert (g[g.index.get_level_values(1).duplicated(keep=False) | True] <= 1).all() or \
       (d.groupby(['source_idperson','hw'])['ils_dispy'].agg(['size','nunique']).query('size>1 and nunique>1').empty)
print('determinism gate PASS')

if not RUN_PRICING:
    assert all((PRICE_DIR2/f'priced_{i:05d}.parquet').exists() or (PRICE_CACHE_SHARED/f'priced_{i:05d}.parquet').exists()
               for i in range(0, len(hh_all), CHUNK)), 'RUN_PRICING=False but a pricing chunk is missing from cache'

## 11. Take-up traits + masked income

**Decision — revealed-first take-up.** Social-assistance receipt is not universal; the entitlement
priced in §10 overstates income for non-takers. We reveal take-up where the observed state settles
it (reported receipt ⇒ taker; entitled-but-not-reported ⇒ non-taker) and impute the rest with a
seeded Bernoulli at the in-sample revealed rates (non-working / working). The mask subtracts the
priced `bsa00_s` from `ils_dispy` for non-takers, per alternative type.

In [ ]:
# take-up traits from the P2a draw-0 (observed) entitlement — a household property
rng_seed = 20162016
d0 = (priced_p2a[(priced_p2a['ruro_decider']==1) & (priced_p2a['draw']==0)]
      [['source_idhh','source_idorighh','bsa00_s']].rename(columns={'bsa00_s':'ent0'}))
tt = d0.merge(singles_dec[['idhh','is_worker']].assign(
        reported=pd.to_numeric(singles_dec.get('bsa00',0), errors='coerce').fillna(0)>0),
      left_on='source_idhh', right_on='idhh', how='left')
tt['entitled0'] = tt['ent0'] > 0
nw, wk = tt[tt.is_worker==0], tt[tt.is_worker==1]
rate_nw = float(nw.loc[nw.entitled0,'reported'].mean())
rate_w  = float(wk.loc[wk.entitled0,'reported'].mean())
print(f'in-sample revealed take-up: nw={rate_nw:.3f} (n={int(nw.entitled0.sum())}), '
      f'w={rate_w:.3f} (n={int(wk.entitled0.sum())})')
tt = tt.sort_values('source_idorighh').reset_index(drop=True)
rng = np.random.default_rng(rng_seed)
bern_nw = rng.random(len(tt)) < rate_nw
bern_w  = rng.random(len(tt)) < rate_w
revealed = tt['entitled0'] | tt['reported']
own_taker = tt['reported']
tt['takeup_nw'] = np.where(tt.is_worker==0, np.where(revealed, own_taker, bern_nw), bern_nw)
tt['takeup_w']  = np.where(tt.is_worker==1, np.where(revealed, own_taker, bern_w),  bern_w)
trait = tt[['source_idhh','takeup_nw','takeup_w']]
print('take-up shares:', trait[['takeup_nw','takeup_w']].mean().round(3).to_dict())

## 12. Assemble engine-ready singles + apply the BPOOL band convention

**Decision — bpool bands (not assembled).** The estimator's hours-band indicators must match the
draw design. The bpool bands are the ones the certified draws were generated under (and the ones
the certified reference used), so they keep the trial certified-comparable. The draw-0 occupation
fix-up mode-imputes the handful of observed workers with unknown occupation (`loc4=-2`) — simulated
working draws are already sampled into `{1,2,3,4}` by the empirical occupation draw.

The DE-origin `assemble_singles` zero-stubs the FR regional covariates; §12b revives them before the
band overwrite so the engine-ready frame is region-live from here on.

In [ ]:
from dclaborsupply_app.de.engine_ready import assemble_singles

# ── P2a-3. Features fixes → take-up mask → assemble (+ band-convention gate) ──
feat2 = draws_p2a.copy()
dagm = pd.to_numeric(singles_dec.set_index('idhh')['dag'], errors='coerce')
feat2['age_norm']  = (feat2['idhh_true'].map(dagm) - float(dagm.mean())) / 10.0
feat2['age_norm2'] = feat2['age_norm']**2
kid = pd.to_numeric(singles_df['dag'], errors='coerce') < CONFIG['age_range'][0]
feat2['n_children'] = feat2['idhh_true'].map(kid.groupby(singles_df['idhh']).sum()).fillna(0).astype('int16')
feat2['source_idhh'] = feat2['idhh_true']
feat2['source_idorighh'] = pd.to_numeric(feat2['idorighh'], errors='coerce').astype('int64')

# Draw-0 occupation fix-up (mirror 17e / continuous.py SAFETY): the certified B-pool
# SAMPLES loc4 in {1,2,3,4} for every simulated working draw, but the chosen row
# (draw 0) keeps the OBSERVED occupation, which is -2 for unknown-occupation workers.
# assemble_singles rejects working rows with loc4 not in {1,2,3,4}; mode-impute them.
_h0 = pd.to_numeric(feat2['hours'], errors='coerce').fillna(0.0)
_l40 = pd.to_numeric(feat2['loc4'], errors='coerce').fillna(-2).astype(int)
_bad0 = (_h0 > 0) & ~_l40.isin([1, 2, 3, 4])
if _bad0.any():
    assert (pd.to_numeric(feat2.loc[_bad0, 'draw'], errors='coerce') == 0).all(), 'expected only draw-0 rows; simulated working draws are sampled in {1,2,3,4}'
    _pool0 = pd.to_numeric(singles_dec.loc[(pd.to_numeric(singles_dec['is_worker'], errors='coerce') == 1) & pd.to_numeric(singles_dec['loc4'], errors='coerce').isin([1, 2, 3, 4]), 'loc4'], errors='coerce').astype(int).to_numpy()
    _mode0 = int(np.argmax(np.bincount(_pool0, minlength=5)[1:5]) + 1)
    feat2.loc[_bad0, 'loc4'] = _mode0
    print(f'draw-0 unknown-occupation workers mode-imputed: {int(_bad0.sum())} row(s) -> loc4={_mode0}')

pre_bands = feat2[['source_idhh','draw','working_pt1','working_pt2','working_ft','working_lh']].copy()

# take-up mask on the new pricing (same traits, same arithmetic)
pm2 = (priced_p2a.merge(alt2[['source_idhh','draw','hours']].drop_duplicates(), on=['source_idhh','draw'])
                 .merge(trait, on='source_idhh', how='left'))
take = np.where(pm2['hours']>0, pm2['takeup_w'], pm2['takeup_nw']).astype(bool)
pm2['ils_dispy_takeup'] = pd.to_numeric(pm2['ils_dispy']) - pd.to_numeric(pm2['bsa00_s']).fillna(0)*(~take)

er_p2a = assemble_singles(pm2, feat2, alt_keys=('draw',), income_source='ils_dispy_takeup')
print('engine-ready P2a:', er_p2a.shape, '| chosen:', int(pd.to_numeric(er_p2a['is_chosen']).sum()))

# BAND-CONVENTION GATE: how many alt-rows did the assembler's recompute reflag?
post = er_p2a[['source_idhh','draw','working_pt1','working_pt2','working_ft','working_lh']]
cmpb = pre_bands.merge(post, on=['source_idhh','draw'], suffixes=('_bpool','_asm'))
for b in ['working_pt1','working_pt2','working_ft','working_lh']:
    n = int((cmpb[f'{b}_bpool'] != cmpb[f'{b}_asm']).sum())
    print(f'{b}: reflagged rows = {n}')
print('STOP: decide band convention before estimating (bpool = certified-comparable; '
      'assembled = trial-consistent). Report the counts.')


### 12b. Revive region / urbanisation / gsur onto the engine-ready frame

`assemble_singles` (DE origin) zero-stubs `drgn1` (region), `drgur/drgmd/drgru` (urbanisation), and
`gsur` (group unemployment rate). The FR market-opportunity block (`beta_E_gsur`, `beta_E_drgn*`,
`beta_E_drgur/drgmd`) needs their real values; left at zero, `beta_E` absorbs the regional variation.
Reviving them (mapped from `singles_dec` by `idhh`) is what takes the fit from the region-dead
negLL 19071.6562 to the region-live **19053.4655**. Mirrors `fr_data_walkthrough.ipynb` P2a-10.

In [ ]:
# Revive region/urbanisation/gsur onto er_p2a (they are zero-stubbed by assemble_singles).
_src = singles_dec.set_index('idhh')
for c in ['drgn1', 'drgur', 'drgmd', 'drgru', 'gsur']:
    er_p2a[c] = er_p2a['idhh'].map(pd.to_numeric(_src[c], errors='coerce')).astype('float64')
_s = er_p2a.groupby('idhh')[['drgn1', 'drgur', 'drgmd', 'drgru', 'gsur']].first()
assert _s['drgn1'].between(1, 8).all(), 'drgn1 out of [1,8]'
assert (_s[['drgur', 'drgmd', 'drgru']].sum(axis=1) == 1).all(), 'urbanisation not one-hot'
assert _s['gsur'].between(0.05, 0.23).all() and _s['gsur'].nunique() > 1, 'gsur dead/degenerate'
print('region/urb/gsur revived. drgn1 counts:', _s['drgn1'].value_counts().sort_index().to_dict())
print('gsur: mean %.4f  n_unique %d' % (float(_s['gsur'].mean()), int(_s['gsur'].nunique())))

In [ ]:
# BPOOL band overwrite -> er_b (the estimation frame). No assembled-bands fit is kept.
er_b = er_p2a.merge(pre_bands, on=['source_idhh','draw'], suffixes=('', '_bpool'))
for b in ['working_pt1','working_pt2','working_ft','working_lh']:
    er_b[b] = er_b[f'{b}_bpool']; er_b.drop(columns=f'{b}_bpool', inplace=True)
nz = er_b.loc[pd.to_numeric(er_b['working'])==0, ['working_pt1','working_pt2','working_ft','working_lh']]
assert (nz == 0).all().all(), 'bpool flags nonzero on non-working rows'
er_b.to_parquet(NOTEBOOK_OUTPUT_ROOT / 'fr_singles_engine_ready_p2a_bpool.parquet')
print('er_b (bpool bands):', er_b.shape)

## 13. Load the certified spec + warm start; pin 10 params, leave the occupation block free

In [ ]:
from dclaborsupply import EstimationSpec
from dclaborsupply.spec.parser import load_custom_initial_values
from dclaborsupply.data.loader import load_singles
from dclaborsupply.likelihood.engine_jax import build_jax_singles_ll
from dclaborsupply.likelihood.index import compute_index
import jax, jax.numpy as jnp
from scipy.optimize import minimize

SPEC_YAML = MNL_ROOT / 'scripts/bpool/specs/estimation_spec_joint_pooled_v1_bll0_tlmpin.yaml'
THETA_CSV = MNL_ROOT / 'scripts/bpool/specs/theta_hat_realdata_901_v1.csv'
spec  = EstimationSpec.from_yaml(str(SPEC_YAML))
names = list(spec.all_param_names)
raw_theta = load_custom_initial_values(THETA_CSV)
theta0 = np.array([float(raw_theta[n]) for n in names], dtype='float64')
idx = {n: i for i, n in enumerate(names)}
t0 = jnp.asarray(theta0)

PINNED = ['beta_l0_m','beta_l_age_m','beta_l_age2_m','beta_l0_f','beta_l_age_f',
          'beta_l_age2_f','beta_l_nkids_f','theta_l_f','beta_E_y2015','beta_E_y2017']   # 10
OCC = [f'beta_occ_{k}_{s}' for k in (2,3,4) for s in ('m','f')]                         # 6 — FREE in P2a
AT_BOUND = ['beta_l_age2_sm','beta_l_age2_sf']
b4 = list(spec.get_bounds_tuple())
for p in PINNED: b4[idx[p]] = (float(t0[idx[p]]), float(t0[idx[p]]))   # pin the 10; occ stays free
print(f'spec: {len(names)} params | pinned {len(PINNED)} | occ block FREE ({len(OCC)})')

## 14. Fit — 10 pinned, occupation block free

**Why the occupation block is identified now.** Under the trial's `occ_spec='fixed'` every simulated
working draw carried the same imputed occupation, so `beta_occ_*` moved the utility of all working
alternatives together — collinear with `beta_E` (a ridge; the pre-committed arbiter in the source
box-24 confirmed the direction). The certified draws sample occupation empirically, so the six
`beta_occ_*` now shift *relative* alternative utilities and are separately identified.

In [ ]:
if RUN_ESTIMATION:
    # loader data objects from er_b, then L-BFGS-B from the certified warm start (occ free)
    sm_b = er_b[pd.to_numeric(er_b['dgn'])==1].reset_index(drop=True)
    sf_b = er_b[pd.to_numeric(er_b['dgn'])==0].reset_index(drop=True)
    meta_b = {'normalization': {'singles': {'c_scale': float(er_b['c_scale'].iloc[0]),
                                            'l_scale': float(er_b['l_scale'].iloc[0])}}}
    dm_b = load_singles(sm_b, spec, is_male=True,  metadata=meta_b)
    df_b = load_singles(sf_b, spec, is_male=False, metadata=meta_b)
    nm_b, _ = build_jax_singles_ll(dm_b, spec, is_male=True)
    nf_b, _ = build_jax_singles_ll(df_b, spec, is_male=False)
    tot_b = jax.jit(lambda t: nm_b(t) + nf_b(t)); vg_b = jax.jit(jax.value_and_grad(tot_b))
    def f_b(t):
        v, g = vg_b(jnp.asarray(t)); return float(v), np.asarray(g, dtype='float64')
    r_b = minimize(f_b, np.asarray(t0), jac=True, method='L-BFGS-B', bounds=b4,
                   options={'maxiter': 5000, 'maxcor': 30, 'ftol': 1e-15, 'gtol': 1e-10})
    print(f'P2a fit negLL {r_b.fun:.4f}  iters={r_b.nit}  converged={r_b.success}  (target 19053.4655, region-live)')
    assert abs(r_b.fun - 19053.4655) < 1e-2, 'P2a region-live fit did not reproduce the anchor negLL'
    occ_est = {n: round(float(r_b.x[idx[n]]), 3) for n in OCC}
    print('occupation estimates (now free):', occ_est)
else:
    print("[SKIPPED: RUN_ESTIMATION=False] estimation (L-BFGS-B fit) not run in v2 (flip RUN_ESTIMATION to run).")
    r_b = dm_b = df_b = sm_b = sf_b = tot_b = meta_b = None

## 15. Freeze + adapter — write provenance, theta CSV, and the styled-report inputs

In [ ]:
if RUN_ESTIMATION:
    # ── P2a-5. Freeze P2a (bpool bands) + adapter to its own output folder ──────
    import json
    from datetime import datetime, timezone
    OUT2 = NOTEBOOK_OUTPUT_ROOT; OUT2.mkdir(parents=True, exist_ok=True)
    STEM2 = 'fr_p2a_singles2016'

    pd.DataFrame({'param': names, 'certified': np.asarray(t0), 'trial': r_b.x,
                  'moved': np.abs(r_b.x - np.asarray(t0))}
                 ).to_csv(NOTEBOOK_OUTPUT_ROOT / 'theta_p2a_singles_2016_v1.csv', index=False)
    prov2 = {'negll_fit': float(r_b.fun), 'n_hh': 1555, 'n_alts': 101,
             'engine_ready': str(NOTEBOOK_OUTPUT_ROOT / 'fr_singles_engine_ready_p2a_bpool.parquet'),
             'theta_csv': str(NOTEBOOK_OUTPUT_ROOT / 'theta_p2a_singles_2016_v1.csv'),
             'status': 'provisional_notebook_result',
             'pinned': PINNED,                       # 10 only — occupation block is FREE
             'at_bound': ['beta_l_age2_sm', 'beta_l_age2_sf'],
             'band_convention': 'bpool (11342/7391/7541 rows reflagged vs assembled; negLL 19071.6562 vs 19130.4360)',
             'draws_design': 'certified B-pool (D1 hours mixture, W1 occ-conditional wages, empirical occ, pi0=0.10, seed 2026)',
             'repair_note': ('region/urbanisation/gsur revived onto the engine-ready frame before the fit '
                             '(zero-stubbed by assemble_singles); negLL 19071.6562 region-dead -> 19053.4655 '
                             'region-live; beta_E -4.31 -> -2.90 absorbed the regional variation')}
    json.dump(prov2, open(NOTEBOOK_OUTPUT_ROOT / 'p2a_fit_provenance.json', 'w'), indent=2)

    tab = pd.read_csv(prov2['theta_csv'])
    pinned2 = list(prov2['pinned']); assert len(pinned2) == 10
    trial = dict(zip(tab['param'], tab['trial'].astype(float)))
    cert  = dict(zip(tab['param'], tab['certified'].astype(float)))
    free2 = [p for p in tab['param'] if p not in set(pinned2)]            # 37 free
    joint = {'success': True,
             'message': 'P2a fit (singles-only; certified draws; occ block free; bpool bands; region/urb/gsur revived)',
             'final_ll': -float(prov2['negll_fit']),
             'parameters':      {p: float(trial[p]) for p in tab['param']},
             'standard_errors': {p: None for p in free2},
             'initial_values':  {p: float(cert[p]) for p in free2},
             'theta':           [float(trial[p]) for p in free2]}
    data = {'specification': 'joint_pooled_v1_bll0_tlmpin_p2a_singles2016',
            'wage_spec': 'vw', 'timestamp': datetime.now(timezone.utc).isoformat(),
            'command_line': 'notebooks/fr_singles_pipeline_v2.ipynb - P2a active-dev (provisional)',
            'metadata': {'mnl_base': (OUT2/STEM2).as_posix(),
                         'spec_config': 'scripts/bpool/specs/estimation_spec_joint_pooled_v1_bll0_tlmpin.yaml',
                         'group': 'joint', 'n_jobs': -1, 'opt_method': 'notebook P2a (warm->fit)',
                         'analytical_gradient': True, 'strict_validation': False,
                         'produced_by': 'notebooks/fr_singles_pipeline_v2.ipynb',
                         'status': 'provisional_notebook_result',
                         'pinned_params': pinned2, 'at_bound': prov2['at_bound'],
                         'note': '10 pinned (couples+year) excluded from SE/init/theta; occ block estimated; region-live.'},
            'results': {'joint': joint},
            'summary': {'joint_ll': -float(prov2['negll_fit']), 'n_obs_total': 1555, 'n_groups_total': 1555}}
    json.dump(data, open(OUT2/'estimation_results_p2a_singles2016.json', 'w'), indent=1)

    er_b.to_parquet(OUT2/f'{STEM2}__singles.parquet')
    mnlmeta = {'source': 'P2a bpool draws (101 alts/HH)', 'produced_by': 'notebooks/fr_singles_pipeline_v2.ipynb',
               'status': 'provisional_notebook_result',
               'n_draws': {'singles': 101},
               'normalization': {'singles': {'c_scale': float(er_b['c_scale'].iloc[0]),
                                             'l_scale': float(er_b['l_scale'].iloc[0]), 'n_chosen': 1555}},
               'row_counts': {'singles': int(len(er_b)), 'total': int(len(er_b))},
               'cluster_key': {'cluster_id_col': 'cluster_id', 'source_col': 'idorighh'},
               'prior_convention': 'prior=clip(exp(clip(log_prior,-700,700)),1e-16,None)'}
    json.dump(mnlmeta, open(OUT2/f'{STEM2}__mnlmeta.json', 'w'), indent=1)
    print('P2a frozen + adapted:', OUT2)
else:
    print("[SKIPPED: RUN_ESTIMATION=False] freeze + adapter not run in v2 (flip RUN_ESTIMATION to run).")

## 16. Cluster-robust SEs — the occupation block included (35 free params)

In [ ]:
if RUN_INFERENCE and r_b is not None:
    # ── P2a-7. Cluster-robust SEs — occupation block FREE (35 identified) ───────
    # Mirrors box "23. Cluster-robust SEs" but on the P2a bpool-band refit: scores from
    # dm_b/df_b, evaluated at r_b.x, Hessian of tot_b; free block excludes ONLY the 10
    # pinned + 2 at-bound -> 35 free (the 6 occupation params ARE now identified).
    import jax, jax.numpy as jnp
    from dclaborsupply.likelihood.engine_jax import build_jax_singles_ll
    # NOTE (v2, preliminary): this 35-parameter non-bound Hessian and sandwich calculation is
    # preliminary evidence. The strict production verdict separately evaluates all 37 run-level
    # free parameters, bound handling, the regional design rank, the raw regional Hessian block,
    # the conditional Schur complement, persisted gradients, chunked scores and fresh-process reload.

    ll_m, _ = build_jax_singles_ll(dm_b, spec, is_male=True,  per_group=True)
    ll_f, _ = build_jax_singles_ll(df_b, spec, is_male=False, per_group=True)
    th = jnp.asarray(r_b.x)

    S = np.vstack([np.asarray(jax.jacrev(ll_m)(th)), np.asarray(jax.jacrev(ll_f)(th))])  # per-HH scores of +LL
    g_chk = S.sum(axis=0) + np.asarray(jax.grad(tot_b)(th))       # T1: sum(scores) == -grad(negLL)
    print('T1 sign check max|.|:', np.abs(g_chk).max())           # expect ~1e-8

    cl = np.concatenate([sm_b.groupby('idhh', sort=False)['cluster_id'].first().to_numpy(),
                         sf_b.groupby('idhh', sort=False)['cluster_id'].first().to_numpy()])
    assert len(cl) == S.shape[0]
    uc = pd.unique(cl); print('clusters:', len(uc), '(single wave: ≈ one hh per cluster)')
    B = np.zeros((len(names), len(names)))
    for j in uc:
        s = S[cl == j].sum(axis=0); B += np.outer(s, s)           # meat

    H = np.asarray(jax.jacfwd(jax.grad(tot_b))(th))               # bread: Hessian of P2a negLL, 47x47
    AT_BOUND = ['beta_l_age2_sm', 'beta_l_age2_sf']
    free_names = [n for n in names if n not in set(PINNED) | set(AT_BOUND)]   # 35 free — OCC INCLUDED
    print(f'free params: {len(free_names)} (10 pinned + 2 at-bound excluded; occupation block free)')
    fi = [idx[n] for n in free_names]
    Hf, Bf = H[np.ix_(fi, fi)], B[np.ix_(fi, fi)]
    eig = np.linalg.eigvalsh(Hf)
    print(f'H_free: min eig {eig.min():.4f}  cond {eig.max()/max(eig.min(),1e-300):.3e}  (want min>0)')
    Hinv = np.linalg.pinv(Hf)
    V = Hinv @ Bf @ Hinv
    se   = pd.Series(np.sqrt(np.clip(np.diag(V), 0, None)),    index=free_names, name='se_clustered')
    se_h = pd.Series(np.sqrt(np.clip(np.diag(Hinv), 0, None)), index=free_names, name='se_hessian')

    tab = pd.DataFrame({'estimate': [float(r_b.x[idx[n]]) for n in free_names]},
                       index=free_names).join([se_h, se])
    tab['t_robust'] = tab['estimate'] / tab['se_clustered']
    tab['ratio_cl_h'] = tab['se_clustered'] / tab['se_hessian']
    occ_rows = [n for n in free_names if n.startswith('beta_occ_')]
    print('\nOCC block (now free — the point of P2a):'); print(tab.loc[occ_rows].round(4).to_string())
    print('\nT5 ratio med/max:', tab['ratio_cl_h'].median().round(3), tab['ratio_cl_h'].max().round(3))
    tab.to_csv(NOTEBOOK_OUTPUT_ROOT / 'p2a_se_clustered.csv')

    # inject SEs into the P2a results JSON so a P2a-6 re-run renders t/p for the free block
    _p = str(NOTEBOOK_OUTPUT_ROOT / 'estimation_results_p2a_singles2016.json')
    d = json.load(open(_p))
    d['results']['joint']['standard_errors'] = {n: float(se[n]) for n in free_names}
    d['metadata']['se_method'] = ('cluster-robust sandwich, analytic per-household scores '
                                  '(jacrev of per_group +LL), cluster=idorighh, free block = '
                                  f'{len(free_names)} params (10 pinned + 2 at-bound excluded; occ free)')
    json.dump(d, open(_p, 'w'), indent=1)
    print('SEs injected into P2a JSON — rerun P2a-6 once more so t/p columns render.')

else:
    print("[SKIPPED: RUN_INFERENCE and r_b is not None=False] inference (cluster-robust SEs) not run in v2 (flip RUN_INFERENCE and r_b is not None to run).")

## 17. Inline post-estimation report — the hours-distribution figure is the rebuild verdict

In [ ]:
if RUN_POST_ESTIMATION:
    # wrapped so a headless report failure does not abort the pipeline / anchor
    try:
        # ── P2a-6. Post-estimation report — INLINE (P2a singles; no HTML kept) ──────
        # Runs RURO_post_estimation_styled.py into the trial folder, then displays the
        # LLM summary + every diagnostic image right here. The *_llm_summary_*.md stays
        # in outputs/p2a_singles2016/notebook_dev_v2/; the HTML is discarded (we read it inline).
        import subprocess, sys, glob, os
        from pathlib import Path
        from IPython.display import Image, Markdown, display
    
        candidates = [Path.cwd(), Path.cwd().parent]
        MNL_ROOT = next((p for p in candidates if (p / 'scripts/enhanced/RURO_post_estimation_styled.py').exists()), None)
        if MNL_ROOT is None:
            raise FileNotFoundError('Could not locate MNL repo root (scripts/enhanced/RURO_post_estimation_styled.py).')
    
        OUTDIR = MNL_ROOT / 'outputs/p2a_singles2016/notebook_dev_v2'
        PREFIX = 'p2a_singles2016_'
    
        # clean only PRIOR report artifacts (prefix-scoped) so nothing stale is re-displayed;
        # inputs (fr_p2a_singles2016__*, estimation_results_*.json) do NOT match this prefix.
        for pat in (f'{PREFIX}*.png', f'{PREFIX}*.html', f'{PREFIX}llm_summary_*.md'):
            for f in OUTDIR.glob(pat):
                f.unlink()
    
        cmd = [
            sys.executable, 'scripts/enhanced/RURO_post_estimation_styled.py',
            '--results-json', 'outputs/p2a_singles2016/notebook_dev_v2/estimation_results_p2a_singles2016.json',
            '--mnl-base',     'outputs/p2a_singles2016/notebook_dev_v2/fr_p2a_singles2016',
            '--spec-config',  'scripts/bpool/specs/estimation_spec_joint_pooled_v1_bll0_tlmpin.yaml',
            '--output-dir',   'outputs/p2a_singles2016/notebook_dev_v2',   # PNGs land flat in the trial folder
            '--no-spec-subdir',                              # ...not a spec subdir
            '--llm-summary-dir', 'outputs/p2a_singles2016/notebook_dev_v2',
            '--prefix',       PREFIX,
            '--cluster-col',  'idorighh',
        ]
        print('running:', ' '.join(cmd))
        r = subprocess.run(cmd, cwd=MNL_ROOT, capture_output=True, text=True)
        tail = '\n'.join((r.stdout + r.stderr).splitlines()[-15:])
        print(tail)
        if r.returncode != 0:
            raise RuntimeError(f'post-estimation failed (exit {r.returncode}) — full stderr above')
    
        # discard the HTML (we render everything inline) and the couples PNGs (no couples
        # data in this P2a singles -> pure redundancy, don't leave them on disk either)
        for h in OUTDIR.glob(f'{PREFIX}*.html'):
            h.unlink()
            print('removed HTML:', h.name)
        for cp in list(OUTDIR.glob(f'{PREFIX}cou_*.png')) + list(OUTDIR.glob(f'{PREFIX}couples*.png')):
            cp.unlink()
    
        # 1) LLM summary markdown, inline
        md_files = sorted(OUTDIR.glob(f'{PREFIX}llm_summary_*.md'), key=os.path.getmtime)
        if md_files:
            print(f'\nkept summary: {md_files[-1].relative_to(MNL_ROOT)}')
            display(Markdown(md_files[-1].read_text(encoding='utf-8')))
        else:
            print('WARNING: no *_llm_summary_*.md produced')
    
        # 2) diagnostic images inline, deduped + sorted. Singles trial -> DROP the couples
        # plots (cou_m/cou_f *_mu, *_contours): they are driven only by the pinned couples
        # params, have no couples microdata behind them, and are pure redundancy here.
        def _is_couples(stem):
            s = stem[len(PREFIX):]
            return s.startswith('cou_') or s.startswith('couples') or '_couples' in s
        allpng = sorted({p.resolve() for p in OUTDIR.glob(f'{PREFIX}*.png')}, key=lambda p: p.name)
        pngs = [p for p in allpng if not _is_couples(p.stem)]
        dropped = [p.stem[len(PREFIX):] for p in allpng if _is_couples(p.stem)]
        if dropped:
            print(f'\nsuppressed {len(dropped)} couples plot(s) (no couples data in this P2a singles run): {dropped}')
        print(f'{len(pngs)} singles diagnostic image(s):')
        for p in pngs:
            display(Markdown(f'**{p.stem[len(PREFIX):]}**'))
            display(Image(filename=str(p)))
    except Exception as _e:
        print('report step skipped:', type(_e).__name__, _e)
else:
    print("[SKIPPED: RUN_POST_ESTIMATION=False] post-estimation report not run in v2 (flip RUN_POST_ESTIMATION to run).")

## 18. Master record + regression anchor

In [ ]:
if RUN_ESTIMATION and r_b is not None:
    # ── P2a-8. P2a master record + regression anchor (target 19053.4655, region-live) ────────
    # Clones the trial's master-record (box 25) and anchor (box 27) with P2a numbers,
    # all read from p2a_fit_provenance.json (no hardcoded diagnostics).
    import json
    from dclaborsupply.data.loader import load_singles
    from dclaborsupply.likelihood.index import compute_index
    # NOTE (v2): this is a same-kernel disk-reload check, not the formal fresh-process
    # production cold-reload gate (that gate lives in MNL/scripts/p2a/).

    _prov  = json.load(open(NOTEBOOK_OUTPUT_ROOT / 'p2a_fit_provenance.json'))
    _negll = float(_prov['negll_fit'])
    _theta_c = float(r_b.x[idx['theta_c_singles']])

    md = f"""# FR 2016 Singles RURO — P2a Rebuild Master Record
    Sample: 1,555 single households (20-60), FR_2016_a3, certified B-pool draws ({_prov['n_alts']} alts/HH).
    negLL fit **{_negll:.4f}** (37 free / 10 pinned / 2 at-bound; occupation block ESTIMATED, not pinned).
    ## What changed vs the trial (negLL 4106.6042; occ pinned; uniform proposals)
    1. Draws design: {_prov['draws_design']}.
    2. Band convention: {_prov['band_convention']}.
    3. Occupation block FREE and identified; region/urbanisation/gsur revived from zero-stubs (negLL 19071.6562 -> 19053.4655; beta_E absorbed the regional variation).
    4. theta_c_singles = {_theta_c:.4f}.
    ## Verdict artifacts (outputs/p2a_singles2016/)
    - P2a-6 inline report: the hours-distribution figure is the rebuild verdict (target: the 33.5-36.5 band recovering toward the observed ~25%).
    - P2a-7 SEs: p2a_se_clustered.csv (sandwich, analytic scores, cluster=idorighh; occupation-block SEs included).
    - results JSON + *_llm_summary_*.md + PNGs in the same folder.
    ## Provenance
    {_prov['theta_csv']} | p2a_fit_provenance.json | {_prov['engine_ready']}
    """
    open(NOTEBOOK_OUTPUT_ROOT / 'P2A_MASTER_RECORD.md', 'w').write(md)
    print('wrote outputs/p2a_singles2016/P2A_MASTER_RECORD.md')

    # ── regression anchor — cold reload from provenance, numpy backend ──
    _tab  = pd.read_csv(_prov['theta_csv']).set_index('param')['trial']
    _er   = pd.read_parquet(_prov['engine_ready'])
    _meta = {'normalization': {'singles': {'c_scale': float(_er['c_scale'].iloc[0]),
                                           'l_scale': float(_er['l_scale'].iloc[0])}}}
    _dm = load_singles(_er[pd.to_numeric(_er['dgn'])==1].reset_index(drop=True), spec, is_male=True,  metadata=_meta)
    _df = load_singles(_er[pd.to_numeric(_er['dgn'])==0].reset_index(drop=True), spec, is_male=False, metadata=_meta)
    _th = np.array([float(_tab[n]) for n in spec.all_param_names])
    _nll = float(compute_index(spec, (_dm, _df, None), _th, ruro=True, backend='numpy'))
    print(f'anchor negLL = {_nll:.6f}  (target 19053.4655, region-live)')
    assert abs(_nll - 19053.4655) < 1e-2, 'P2a ANCHOR BROKEN — environment/package/data changed'
    print('P2a ANCHOR HOLDS — notebook self-verified.')

else:
    print("[SKIPPED: RUN_ESTIMATION and r_b is not None=False] master record + anchor (disk-reload check) not run in v2 (flip RUN_ESTIMATION and r_b is not None to run).")

## 19. Welfare — V_i^IS + W-measure family (P3-1)

> **PROVISIONAL P2a WELFARE — NOT CERTIFIED AND NOT SAFE FOR MANUSCRIPT USE.**
> These welfare outputs derive from the provisional region-live P2a fit, not the certified
> 47-parameter pooled JMP baseline. Development diagnostics only.

Produce, on this region-live P2a baseline, the ex-ante attained utility `V_i^IS` (household inclusive
value `log Σ_j exp(V_j)`, normalized `V_actual = V_i^IS − log(101)`) and the welfare-measure family
**W1/W3/W4/W6**, plus the survey-weighted inequality battery (Gini, CV², Theil-L, Atkinson). This runs
the two committed tools — `scripts/welfare/stage_p2a_singles_welfare.py` (stages the frozen baseline
into the loader layout: `stacked_hh_uid=idhh`, `year_tag=2`) and `scripts/welfare/run_p2a_singles_welfare.py`
(the measures + inequality) — then verifies the outputs are region-live. Outputs land under
`<EUROMOD-STORAGE>/outputs/welfare/p2a_singles2016/`.

In [ ]:
if RUN_WELFARE:
    # Run the welfare pair on the frozen region-live baseline (cwd is the repo root, set in cell 1).
    print('PROVISIONAL P2a WELFARE — NOT CERTIFIED AND NOT SAFE FOR MANUSCRIPT USE')
    # NOTE (v2): the committed welfare scripts target the shared/production welfare store, not
    # NOTEBOOK_OUTPUT_ROOT; running them here is a provisional dev convenience, not an isolated run.
    import subprocess, sys, os
    for _script in ('scripts/welfare/stage_p2a_singles_welfare.py',
                    'scripts/welfare/run_p2a_singles_welfare.py'):
        print('=== running', _script, '===')
        _r = subprocess.run([sys.executable, _script], cwd=os.getcwd(), capture_output=True, text=True)
        print('\n'.join((_r.stdout + _r.stderr).splitlines()[-16:]))
        if _r.returncode != 0:
            raise RuntimeError(f'{_script} failed (exit {_r.returncode}) — see output above')
else:
    print("[SKIPPED: RUN_WELFARE=False] welfare pair (staging + measures) not run in v2 (flip RUN_WELFARE to run).")

In [ ]:
if RUN_WELFARE:
    # P3-1 verification: confirm the welfare run consumed the region-live baseline.
    import json, pandas as pd
    from pathlib import Path

    d = json.load(open('outputs/p2a_singles2016/estimation_results_p2a_singles2016.json'))
    ll = d['results']['joint']['final_ll']
    print('results JSON final_ll:', ll,
          '->', 'REGION-LIVE (correct)' if abs(ll + 19053.4655) < 0.01 else 'REGION-DEAD (stale!)')

    STORE = Path(r'C:\Users\hisham\MNL\EUROMOD-STORAGE')
    staged = pd.read_parquet(STORE / 'new_data' / 'fr_p2a_singles2016_welfare__singles.parquet',
                             columns=['idhh', 'drgn1', 'gsur'])
    print('staged parquet region-live?', staged['drgn1'].nunique() > 1, '| gsur live?', staged['gsur'].nunique() > 1)

    man = json.load(open(STORE / 'outputs/welfare/p2a_singles2016/run_manifest_p2a_v1.json'))
    print('welfare theta_hash:', man['theta_hash'], '| ALL_PASS:', man['verification']['ALL_PASS'])
    ineq = json.load(open(STORE / 'outputs/welfare/p2a_singles2016/inequality_p2a_v1.json'))
    for W in ('W1', 'W4', 'W6'):
        p = ineq['pooled_headline'][W]
        print(f'  {W}: weighted mean {p["weighted_mean"]:.0f} EUR | Gini {p["gini"]:.4f}')
else:
    print("[SKIPPED: RUN_WELFARE=False] welfare verification not run in v2 (flip RUN_WELFARE to run).")

## 20. On the trial anchor (dropped)

The exploratory notebook carried a **second** regression anchor at trial negLL **4106.604220**. Its
inputs — `fr_singles_engine_ready_v5.parquet` and `trial_fit_provenance.json` — come from the
**superseded** path (uniform draws, occupation block pinned, assembled bands), which this clean
notebook deliberately does not produce (Rule 3). Per the task rule, only the **P2a anchor** (§18) is
kept here. To re-verify the trial anchor, run the historical `fr_data_walkthrough.ipynb`.

## Appendix — dropped cells (from `fr_data_walkthrough.ipynb`)

Every source cell not carried here, with the reason. "SUPERSEDED" = belongs to a superseded
estimation round (uniform draws / occ-pinned fit / assembled bands / trial engine-ready v-chain)
and must not survive (Rule 3). "Diagnostic" = one-off probe whose conclusion is folded into a kept
cell or a decision markdown. "Folded" = logic kept, but merged into a single clean cell.

| src cell | disposition | reason |
|---|---|---|
| 28 | dropped | API-discovery `show_api` probe — replaced by explicit imports |
| 29 | dropped | `generate_draws_long` probe on 5 rows |
| 30 | dropped | **SUPERSEDED** — uniform-proposal draws (replaced by certified B-pool, §9) |
| 31 | dropped | superseded — canonicalize the uniform draws |
| 32 | dropped | diagnostic — yem00/yemxp 35h split probe (finding folded into `fr_earnings_policy`) |
| 33 | folded | FR earnings policy → pricing-setup cell (§10); verification prints dropped |
| 34 | folded | baseline/single_idhh → pricing-setup (§10); the 5-hh smoke test dropped |
| 35 | dropped | diagnostic — batch-relativity per-draw sweep |
| 36 | folded | pricing SETUP (baseline_full/runner_full/policy_v2/hh_all/CHUNK/KEEP) → §10; the trial pricing LOOP dropped (clean prices the P2a draws once) |
| 37 | dropped | diagnostic — revealed take-up crosstab (rates folded into §11) |
| 38 | dropped | diagnostic — bsaot probe |
| 39 | dropped | inspection — `assemble_singles` source print |
| 40 | folded | take-up traits + mask → §11 (recomputed on `priced_p2a` draw 0) |
| 41 | dropped | inspection — assembly-contract print |
| 42 | dropped | inspection — contract/provenance resolve |
| 43 | dropped | **SUPERSEDED** — trial `feat` engine-ready features |
| 44 | dropped | **SUPERSEDED** — trial `assemble_singles` (engine-ready v1) |
| 45 | folded | spec load → §13 (kept: spec/names/warm start; dropped: exploratory attr prints) |
| 46 | dropped | **SUPERSEDED** — engine-ready v2 region revival |
| 47 | folded | gsur construction → §8 (the `singles_dec` merge; the `er_singles` overwrite dropped) |
| 48 | dropped | diagnostic — negLL flatness/liveness probe on the v-chain |
| 49 | dropped | **SUPERSEDED** — engine-ready v4 |
| 50 | dropped | **SUPERSEDED** — engine-ready v5-partial |
| 51 | folded | pexp construction → §8 |
| 52 | dropped | **SUPERSEDED** — engine-ready v5 + trial-baseline probe |
| 53 | dropped | **SUPERSEDED** — trial fit v1 (unpinned) |
| 54 | dropped | **SUPERSEDED** — trial fit v2 (occupation block pinned) |
| 55 | dropped | diagnostic — gradient-mask probe |
| 56 | dropped | **SUPERSEDED** — trial polish fit |
| 57–63 | dropped | **SUPERSEDED** — trial adapter/freeze/report/provenance-guard |
| 64 | dropped | **SUPERSEDED** — trial cluster-robust SEs (P2a SEs kept, §16) |
| 65 | dropped | diagnostic — box-24 conditional-wage arbiter (verdict cited in §14) |
| 66 | dropped | **SUPERSEDED** — trial master record (P2a record kept, §18) |
| 67 | dropped | **SUPERSEDED** — trial verdict patch |
| 68 | dropped | trial regression anchor — inputs not produced here (see §19) |
| 73 | dropped | **SUPERSEDED** — P2a-4 assembled-bands fit (bpool bands kept, §12) |
| 74 | folded | kept the `er_b` bpool-band build (§12); dropped the assembled-vs-bpool fit comparison |

Kept verbatim/adapted: data read, schema, config, classification, funnel 4.0–4.6, features+loc4,
observed pricing (guarded), singles restriction, folded features (§8), P2a-0/P2a-1 draws (§9),
pricing (§10), take-up (§11), assembly+bpool bands (§12), spec (§13), fit (§14), adapter (§15),
SEs (§16), report (§17), record+anchor (§18).